# Project 02 · Customer Churn AI
## EDA → Feature Engineering → Feature Selection → Model Comparison → Hyperparameter Tuning → Final Model


데이터셋: **IBM Telco Customer Churn** (`ai-server/data/ibm_telco.csv`)

Target: **Churn Label** (고객이 이탈했는가? Yes / No) → Classification(분류) 문제



```text
EDA
  ↓
Hypothesis (가설 수립)
  ↓
Feature Engineering (파생변수 생성)
  ↓
Feature Selection
  ↓
Train / Test Split
  ↓
Model Comparison (Logistic Regression, Decision Tree, Random Forest, XGBoost)
  ↓
Hyperparameter Tuning
  ↓
Final Model
  ↓
Research Report
```


---
# Chapter 1. 프로젝트 소개


## Customer Churn이란?

**Customer Churn(고객 이탈)** 이란 고객이 더 이상 우리 서비스(통신사, 구독 서비스 등)를
이용하지 않고 떠나는 것을 의미합니다.

통신사(Telco) 입장에서 고객 한 명을 잃는 것은 단순히 "고객 한 명이 줄어드는 것" 이상의 의미를
가집니다.

- 새로운 고객을 데려오는 비용(마케팅비)은 기존 고객을 유지하는 비용보다 훨씬 비쌉니다.
- 이탈한 고객은 대부분 경쟁사로 이동하기 때문에, 매출 손실이 곧바로 경쟁사의 이익이 됩니다.
- 이탈 직전 고객을 미리 찾아낼 수 있다면, 할인/혜택 등으로 이탈을 막을 수 있는 "골든 타임"이
  생깁니다.

그래서 "누가 이탈할 가능성이 높은가?"를 미리 예측하는 모델은 실무에서 매우 중요하게 사용됩니다.



## 머신러닝 Workflow


```text
데이터 수집
    ↓
EDA (탐색적 데이터 분석)
    ↓
Hypothesis (가설 수립)
    ↓
Feature Engineering (파생변수 생성)
    ↓
Feature Selection (불필요한 Feature 제거)
    ↓
Train / Test Split
    ↓
Model Comparison (여러 모델 비교)
    ↓
Hyperparameter Tuning (최적 파라미터 탐색)
    ↓
Final Model (최종 모델 선정 및 저장)
```

## Feature Engineering의 목적

원본 데이터의 컬럼(예: `Tenure in Months`, `Monthly Charge`)을 그대로 쓰는 것보다,
컬럼들을 조합하거나 가공해서 **모델이 패턴을 더 쉽게 학습할 수 있는 형태**로 바꿔주는 작업입니다.

예를 들어 "월 단위 계약이면서 동시에 신규 고객인 경우" 처럼 두 조건이 겹칠 때만 나타나는
위험 신호는, 원본 컬럼 두 개를 각각 보는 것보다 하나의 파생변수로 만들어주면 모델이 훨씬
더 쉽게 학습합니다.

## Feature Selection의 목적

Feature Engineering으로 파생변수를 많이 만들다 보면, 오히려 서로 비슷한 정보를 담은
중복 Feature가 늘어나거나, Target과 거의 관련이 없는 Feature가 섞이게 됩니다.

Feature Selection은 이렇게 늘어난 Feature 중에서

- 분산이 없어 정보량이 없는 Feature
- 다른 Feature와 거의 동일한 정보를 담은(높은 상관관계) Feature
- Target 예측에 통계적으로 기여하지 않는 Feature



---
# Chapter 2. EDA (Exploratory Data Analysis, 탐색적 데이터 분석)


## EDA를 왜 하는가?

머신러닝 모델은 "데이터를 있는 그대로" 학습합니다. 그래서 데이터에 문제가 있으면
(결측치, 이상치, 잘못 입력된 값 등) 모델도 잘못된 패턴을 학습하게 됩니다.

EDA는 본격적으로 모델을 학습시키기 **전에**, 데이터가 어떻게 생겼는지 먼저 이해하는
과정입니다. "요리를 하기 전에 재료 상태를 먼저 확인하는 것"과 같습니다.

EDA를 통해 우리는 다음 질문에 답할 수 있어야 합니다.

- 이 데이터는 몇 명의 고객, 몇 개의 정보(컬럼)로 이루어져 있는가?
- 비어있는 값(결측치)은 없는가? 있다면 왜 비어있는가?
- 중복된 고객 데이터는 없는가?
- 숫자형 변수들은 어떤 분포를 가지는가? 한쪽으로 치우쳐(Skewed) 있지는 않은가?
- 범주형 변수들은 어떤 값을 가장 많이 가지는가?
- Target(Churn Label)과 각 Feature는 어떤 관계가 있는가?

이제 실제 데이터를 불러오겠습니다.

In [ ]:
# ---------------------------------------------------------------
# 라이브러리 불러오기
# ---------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline  

# 1. seaborn 그래프 기본 스타일
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.unicode_minus"] = False

# 2. OS별 한글 폰트 설정
system_name = platform.system()

if system_name == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system_name == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:  # Linux, Colab, Ubuntu
    plt.rcParams["font.family"] = "NanumGothic"

# 3. 기타 Matplotlib 설정
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

pd.set_option("display.max_columns", 100)

In [ ]:
# ---------------------------------------------------------------
# 데이터 불러오기
# 이 Notebook은 ai-server/notebooks/ 안에 있으므로,
# 데이터 파일은 한 단계 위(../data)에 있습니다.
# ---------------------------------------------------------------
DATA_PATH = "../data/ibm_telco.csv"

df = pd.read_csv(DATA_PATH)

print("데이터 로드 완료")
print("행(Row) 개수, 열(Column) 개수 :", df.shape)

## 2-1. Shape 확인

`df.shape`는 `(행 개수, 열 개수)` 형태의 튜플을 반환합니다.



In [ ]:
# TODO: 데이터프레임 df의 shape(행 개수, 열 개수)을 확인하는 코드를 작성하세요.


## 2-2. info() 확인


`df.info()` 는 다음 정보를 한 번에 보여줍니다.

- 각 컬럼의 이름
- 각 컬럼의 **dtype(데이터 타입)**: 숫자형(`int64`, `float64`)인지 문자형(`object`)인지
- **결측치가 아닌 값의 개수(Non-Null Count)**: 이 값이 전체 행 개수(7,043)보다 작으면
  결측치가 있다는 뜻입니다.
- **Memory Usage(메모리 사용량)**: 데이터가 차지하는 메모리 크기


In [ ]:
# TODO: df.info()를 사용해 각 컬럼의 dtype과 결측치 정보를 확인하세요.


## 2-3. describe() - 5수 요약(Five-Number Summary)

`df.describe()` 는 숫자형 컬럼에 대해 다음 통계량을 보여줍니다.

| 통계량 | 의미 |
|---|---|
| count | 값이 있는 데이터 개수 |
| mean | 평균 |
| std | 표준편차 (데이터가 평균으로부터 얼마나 퍼져 있는가) |
| min | 최솟값 |
| 25% | 하위 25%가 이 값보다 작음 (1사분위수) |
| 50% | 중앙값 (Median) |
| 75% | 하위 75%가 이 값보다 작음 (3사분위수) |
| max | 최댓값 |


In [ ]:
# TODO: df.describe()를 사용해 숫자형 컬럼의 5수 요약 통계를 확인하세요.


In [ ]:
# 문자형(범주형) 컬럼의 요약 통계도 함께 확인합니다.
# top: 가장 많이 등장한 값, freq: 그 값이 등장한 횟수
# TODO: describe()에 include="object" 옵션을 사용해 범주형 컬럼 요약을 확인하세요.


## 2-4. 결측치(Missing Value)

**결측치(Missing Value)** 는 데이터에 값이 채워지지 않고 비어있는 상태(`NaN`)를 말합니다.



In [ ]:
# TODO: 컬럼별 결측치 개수를 구하고, 결측치가 있는 컬럼만 골라 개수가 많은 순으로 정렬해 출력하세요.
# 힌트: df.isnull().sum() -> 0보다 큰 값만 필터링 -> sort_values(ascending=False)
# 결측치 비율(%)도 함께 출력해보세요. (개수 / len(df) * 100)


In [ ]:
# 결측치 위치를 heatmap으로 시각화합니다. (노란색 = 결측치가 있는 위치)
# TODO: sns.heatmap()과 df.isnull()을 사용해 결측치 위치를 시각화하세요.
# 힌트: sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.figure(figsize=(12, 5))

plt.title("Missing Value Heatmap (노란색 = 결측치)")
plt.xlabel("Columns")
plt.ylabel("Row Index")
plt.show()


## 2-5. 중복 데이터(Duplicate)

중복 데이터는 크게 두 가지로 확인합니다.

1. **완전히 동일한 행(Row)이 중복되는 경우**
2. **고객 식별자(Customer ID)만 중복되는 경우** (같은 고객이 두 번 이상 기록된 경우)


In [ ]:
# TODO: 완전히 동일한 행이 몇 개인지, Customer ID가 중복된 개수는 몇 개인지 확인하세요.
# 힌트: df.duplicated().sum(), df["Customer ID"].duplicated().sum()


# TODO: drop_duplicates()로 중복을 제거했을 때 데이터 크기가 어떻게 바뀌는지 확인하세요.


## 2-6. Target 분석 (Churn Label)

`Churn Label` 값을 분석에 사용하기 위해 `Yes` → 1, `No` → 0 으로 변환합니다.
이후 이탈 고객 수, 잔류 고객 수, 이탈률, 클래스 비율을 확인합니다.


In [ ]:
# TODO: Churn Label(Yes/No)을 숫자(1/0)로 변환해 새로운 컬럼 df["Churn"]을 만드세요.
# 힌트: df["Churn Label"].map({"Yes": 1, "No": 0})

print("이탈(1) / 잔류(0) 고객 수:")
print(df["Churn"].value_counts())

print("\n이탈(1) / 잔류(0) 비율:")
print(df["Churn"].value_counts(normalize=True).round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# TODO: axes[0]에 Churn Label의 countplot을 그리세요.
# 힌트: sns.countplot(x="Churn Label", data=df, ax=axes[0], order=["No", "Yes"])


axes[0].set_title("Customer Churn Count (고객 이탈 여부 인원수)")
axes[0].set_xlabel("Churn Label")
axes[0].set_ylabel("Count")

# TODO: axes[1]에 Churn Label 비율을 나타내는 Pie Chart를 그리세요.
# 힌트: churn_ratio = df["Churn Label"].value_counts()
#       axes[1].pie(churn_ratio, labels=churn_ratio.index, autopct="%.2f%%", startangle=90)


axes[1].set_title("Customer Churn Ratio (고객 이탈 비율)")

plt.tight_layout()
plt.show()


## 2-7. 숫자형 변수(Numerical Feature) 분석

각 숫자형 변수에 대해 Histogram(분포), 왜도(Skewness), 첨도(Kurtosis)를 확인합니다.

- **왜도(Skewness)**: 분포가 좌우 대칭인지, 한쪽으로 치우쳐(꼬리가 긴 방향으로) 있는지를
  나타내는 값입니다. 0에 가까울수록 좌우 대칭이고, 양수이면 오른쪽 꼬리가 길고(오른쪽에
  소수의 큰 값), 음수이면 왼쪽 꼬리가 깁니다.
- **첨도(Kurtosis)**: 분포가 얼마나 뾰족한지(중심에 값이 몰려 있는지)를 나타냅니다.
  0에 가까우면 정규분포와 비슷한 뾰족함이고, 음수이면 정규분포보다 완만(납작)합니다.


In [ ]:
numeric_columns = [
    "Age",
    "Tenure in Months",
    "Monthly Charge",
    "Total Charges",
    "Avg Monthly GB Download",
    "Avg Monthly Long Distance Charges",
    "Number of Referrals",
    "Satisfaction Score",
    "CLTV",
    "Total Revenue",
]

# TODO: numeric_columns의 각 컬럼에 대해 히스토그램(sns.histplot, kde=True)을 그리세요.
# 힌트: fig, axes = plt.subplots(5, 2, figsize=(14, 22))로 5x2 격자를 만들고
#       axes = axes.flatten() 후 for i, col in enumerate(numeric_columns): 로 반복하며
#       axes[i]에 그립니다.


plt.tight_layout()
plt.show()


In [ ]:
# TODO: numeric_columns에 대한 평균(mean), 표준편차(std), 왜도(skew), 첨도(kurtosis)를
# 하나의 표(DataFrame)로 요약하세요.
# 힌트: df[numeric_columns].mean(), .std(), .skew(), .kurt()


## 2-8. 범주형 변수(Categorical Feature) 분석

각 범주형 변수에 대해 `value_counts()`(값별 개수), Countplot(막대그래프), 그리고
Target(Churn)과의 관계를 함께 확인합니다.


In [ ]:
categorical_columns = [
    "Gender",
    "Senior Citizen",
    "Married",
    "Dependents",
    "Contract",
    "Internet Service",
    "Internet Type",
    "Payment Method",
    "Paperless Billing",
    "Multiple Lines",
    "Referred a Friend",
    "Offer",
]

# TODO: categorical_columns의 각 컬럼에 대해 countplot을 그리세요.
# 힌트: fig, axes = plt.subplots(4, 3, figsize=(18, 20))로 4x3 격자를 만들고
#       order = df[col].value_counts(dropna=True).index 로 막대 순서를 정합니다.


plt.tight_layout()
plt.show()


In [ ]:
# TODO: categorical_columns의 각 컬럼별로 이탈률(Churn 평균)과 개수를 groupby로 확인하세요.
# 힌트: df.groupby(col, dropna=False)["Churn"].agg(["mean", "count"])


## 2-9. 상관관계(Correlation) 분석

숫자형 변수들 사이의 **피어슨 상관계수(Pearson Correlation Coefficient)** 를 계산합니다.
상관계수는 -1 ~ 1 사이의 값을 가지며,

- **1에 가까울수록** 강한 양의 상관관계 (하나가 커지면 다른 하나도 커짐)
- **-1에 가까울수록** 강한 음의 상관관계 (하나가 커지면 다른 하나는 작아짐)
- **0에 가까울수록** 선형적인 관계가 거의 없음



In [ ]:
corr_columns = numeric_columns + ["Churn"]

# TODO: corr_columns에 대한 상관관계 행렬을 계산해 corr_matrix에 저장하고 heatmap으로 시각화하세요.
# 힌트: corr_matrix = df[corr_columns].corr()
#       sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)

plt.figure(figsize=(10, 8))

plt.title("Correlation Heatmap (숫자형 변수 + Churn)")
plt.show()


## 2-10. Box Plot - 이상치(Outlier) 분석


Box Plot(상자 그림)은 데이터의 분포를 다음 5가지 값으로 요약해서 보여줍니다.

```text
최솟값(이상치 제외) ─ 1사분위수(Q1) ─ 중앙값(Median) ─ 3사분위수(Q3) ─ 최댓값(이상치 제외)
```

Q1과 Q3 사이의 거리를 **IQR(사분위 범위, Interquartile Range)** 이라고 하며,
일반적으로 `Q1 - 1.5*IQR` 보다 작거나 `Q3 + 1.5*IQR` 보다 큰 값을 통계적 이상치로 봅니다.



In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

outlier_summary = {}

# TODO: numeric_columns의 각 컬럼에 대해 Box Plot을 그리고,
# IQR 기준(Q1 - 1.5*IQR ~ Q3 + 1.5*IQR을 벗어난 값) 이상치 개수를 outlier_summary에 저장하세요.
# 힌트: q1 = df[col].quantile(0.25), q3 = df[col].quantile(0.75), iqr = q3 - q1


plt.tight_layout()
plt.show()

print("IQR 기준 컬럼별 이상치 개수:")
for col, cnt in outlier_summary.items():
    print(f"  {col} : {cnt}건")


## 2-11. EDA 핵심 발견 사항(EDA Findings)

> **TODO:** 위 2-1 ~ 2-10에서 직접 계산한 수치를 근거로, 아래 형식에 맞춰 스스로 발견한 내용을 정리해보세요.
> 각 Finding에는 반드시 "근거(직접 계산한 수치)"와 "해석"을 함께 적어야 합니다.

**Finding 1. 계약 형태(Contract)와 이탈률의 관계**
근거: (TODO: 2-8에서 계산한 Contract별 이탈률을 적어보세요)
해석: (TODO)

**Finding 2. 가입 기간(Tenure)과 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 3. 인터넷 서비스 종류와 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 4. 부가 서비스(보안/기술지원) 가입 여부와 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 5. 결제 방식과 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 6. 가족 구성(배우자/부양가족)과 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 7. 고령 고객(Senior Citizen)과 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 8. 부가 서비스 가입 개수와 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 9. 프로모션(Offer)과 이탈률의 관계**
근거: (TODO)
해석: (TODO)

**Finding 10. 숫자형 변수 중 Churn과 상관관계가 가장 강한 변수**
근거: (TODO: 2-9 상관관계 분석 결과를 참고하세요)
해석: (TODO)



# Chapter 3. Hypothesis (가설 수립)

> 아래 가설들은 Chapter 2 EDA 결과를 참고하여 세운 것입니다. 각 가설이 실제로 맞는지는
> Chapter 4(Feature Engineering)에서 파생변수를 직접 만들어 검증합니다. 아직 검증 전이므로
> 수치를 미리 적지 않습니다.

**가설 1.** 계약 기간이 짧을수록(Month-to-Month) 고객은 언제든 위약금 없이 떠날 수 있기
때문에 이탈률이 높을 것이다.

**가설 2.** 가입 초기(6개월 이하) 고객은 아직 서비스에 대한 충성도가 쌓이지 않았고,
신규 고객 대상 경쟁사의 공격적인 프로모션에 노출되기 쉬워 이탈률이 높을 것이다.

**가설 3.** 월 요금이 높을수록 고객이 가격에 부담을 느껴 이탈할 가능성이 높을 것이다.

**가설 4.** 온라인 보안, 기술 지원 같은 부가 서비스에 가입한 고객은 서비스에 더 많이
의존하게 되어(락인 효과) 이탈률이 낮을 것이다.

**가설 5.** 자동 결제(계좌 이체, 신용카드)를 사용하는 고객은 결제의 번거로움이 적어
서비스를 계속 유지할 가능성이 높을 것이다.

**가설 6.** 부양가족이나 배우자가 있는 고객은 가족 단위로 서비스를 사용하고 있어
전환 비용(Switching Cost)이 더 크기 때문에 이탈률이 낮을 것이다.

**가설 7.** 계약 기간이 짧으면서 동시에 신규 고객인 경우, 두 위험 요인이 겹쳐서
이탈률이 각각의 개별 효과보다 훨씬 크게 나타날 것이다.

**가설 8.** 월 요금이 높으면서 동시에 월 단위 계약인 고객은 "비싼데 언제든 떠날 수
있는" 상태이므로 이탈 위험이 가장 클 것이다.

**가설 9.** 기술 지원을 받지 못하면서 월 단위 계약인 고객은, 문제가 생겨도 도움을 받지
못한 채 손쉽게 이탈할 수 있어 위험이 클 것이다.

**가설 10.** 부가 서비스 가입 개수가 극단적으로 적거나(0~1개) 많은 경우(8개 이상)
이탈 패턴이 특이하게 나타날 것이므로, 가입 개수를 하나의 파생변수로 만들면 모델이
비선형적인 패턴을 더 쉽게 학습할 수 있을 것이다.


이제 이 가설들을 실제 파생변수(Feature)로 만드는 **Feature Engineering** 을 시작합니다.



# Chapter 4. Feature Engineering (파생변수 생성)


Feature Engineering은 원본 컬럼을 조합하거나 가공하여, 모델이 패턴을 더 쉽게 학습할 수
있는 새로운 입력값을 만드는 과정입니다. 이번 Chapter에서는 아래 11개의 파생변수를
만듭니다. 각 파생변수는 Chapter 2(EDA)와 Chapter 3(가설)에서 확인한 근거를 바탕으로
설계되었습니다.

```text
1. TenureGroup          - 가입 기간 구간화
2. TotalServices        - 부가 서비스 총 가입 개수
3. StreamingCount       - 스트리밍 서비스 가입 개수
4. InternetCount        - 인터넷 부가 서비스 가입 개수
5. AverageCharge        - 가입 기간 대비 평균 요금
6. HighChargeCustomer   - 고요금 고객 여부
7. ContractRisk         - 계약 + 가입 기간 위험 점수
8. SeniorLongTerm       - 고령 + 장기 고객 여부
9. FamilyCustomer       - 가족 고객 여부
10. CustomerValueTier   - CLTV 기반 고객 가치 등급
11. HighRiskCustomer    - 다중 위험 조건 결합 Feature
```

파생변수를 만들 때는 다음 원칙을 지킵니다.

```text
1. 비즈니스 의미가 있어야 한다.
2. 원본 변수보다 이해하기 쉬운 정보를 제공해야 한다.
3. 예측 시점(고객이 가입/이용 중인 시점)에 계산할 수 있어야 한다.
4. Target(Churn Label)을 직접 사용하지 않는다.
5. 데이터 누수(Data Leakage)가 없어야 한다.
```

## 성능 비교를 위한 준비

각 Feature를 만들 때마다 "이 Feature를 추가하기 전 vs 추가한 후" 모델 성능을 비교합니다.



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

RANDOM_STATE = 42

# Target, 식별자, 데이터 누수 컬럼은 Feature 실험에서 제외합니다.
LEAKAGE_OR_ID_COLUMNS = [
    "Customer ID", "Customer Status", "Churn Score",
    "Churn Category", "Churn Reason", "Churn Label", "Churn",
]

# 상수 컬럼(Country, State, Quarter)과 과도하게 세분화된 위치 정보(City, Zip, 위경도)는
# Baseline에서 제외합니다. (2-3에서 확인한 것처럼 정보량이 없거나 범주가 너무 많습니다)
LOCATION_OR_CONSTANT_COLUMNS = ["Country", "State", "Quarter", "City", "Zip Code", "Latitude", "Longitude"]

BASELINE_COLUMNS = [
    c for c in df.columns
    if c not in LEAKAGE_OR_ID_COLUMNS + LOCATION_OR_CONSTANT_COLUMNS
]

print(f"Baseline Feature 개수: {len(BASELINE_COLUMNS)}")
print(BASELINE_COLUMNS)


def quick_evaluate(frame, feature_cols, label):
    """주어진 Feature 목록으로 RandomForest를 빠르게 학습하고 성능을 반환한다.
    (Feature Engineering 실습용 - Chapter 6의 공식 Pipeline과는 별개)
    """
    X = pd.get_dummies(frame[feature_cols], drop_first=False)
    y = frame["Churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    return {
        "Model": label,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "Recall": round(recall_score(y_test, pred), 4),
        "F1": round(f1_score(y_test, pred), 4),
    }


baseline_result = quick_evaluate(df, BASELINE_COLUMNS, "Baseline (원본 Feature)")
feature_experiment_results = [baseline_result]
pd.DataFrame(feature_experiment_results)

## Feature 1. TenureGroup (가입 기간 구간화)

`Tenure in Months`는 1~72 사이의 연속된 숫자입니다. 하지만 EDA(2-7, Finding 2)에서
확인했듯, 이탈률은 "숫자가 커질수록 일정하게 줄어드는" 단순한 선형 관계가 아니라
0~6개월 구간에서 급격히 높고 그 이후 완만하게 낮아지는 패턴을 보입니다. 이런 "구간별
차이"는 트리 기반 모델은 스스로 어느 정도 찾아낼 수 있지만, Logistic Regression처럼
선형 관계를 가정하는 모델에게는 구간 정보를 직접 알려주는 것이 더 도움이 됩니다.

### 생성 이유
가입 기간을 5개 구간(0-6, 7-12, 13-24, 25-48, 49+ 개월)으로 나누어, "생애 주기 단계"
정보를 범주형 변수로 명확하게 표현합니다.


In [ ]:
# TODO: Tenure in Months를 5개 구간(0-6, 7-12, 13-24, 25-48, 49+ 개월)으로 나누어
# df["TenureGroup"] 컬럼을 만드세요.
# 힌트: pd.cut(df["Tenure in Months"], bins=[0, 6, 12, 24, 48, 72],
#              labels=["0-6", "7-12", "13-24", "25-48", "49+"], include_lowest=True)


# TODO: 구간별 이탈률을 tenure_group_churn 변수에 저장하세요 (observed=True 사용).
# 힌트: df.groupby("TenureGroup", observed=True)["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(8, 5))
# TODO: tenure_group_churn을 이용해 TenureGroup별 이탈률을 barplot으로 시각화하세요.

plt.title("TenureGroup별 이탈률 (Churn Rate by Tenure Group)")
plt.xlabel("TenureGroup")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["TenureGroup"]으로 quick_evaluate를 호출해 성능을 확인하고
# feature_experiment_results 리스트에 결과를 추가하세요.


## Feature 2. TotalServices (부가 서비스 총 가입 개수)

이 데이터셋에는 9개의 서비스 가입 여부 컬럼(Phone Service, Multiple Lines, Online
Security, Online Backup, Device Protection Plan, Premium Tech Support, Streaming TV,
Streaming Movies, Streaming Music)이 각각 별도 컬럼으로 존재합니다. 이를 각각 따로
보는 대신, "총 몇 개의 서비스에 가입했는가"라는 하나의 숫자로 요약하면 고객의
전반적인 서비스 이용 강도를 파악할 수 있습니다. 여러 개의 Yes/No 컬럼을 하나의 숫자형 Feature로 요약하는 방법을 익혀봅니다.

### 생성 이유
Finding 8에서 확인했듯, 서비스 가입 개수와 이탈률은 단순하지 않은(비선형) 관계를
보였습니다. 이 관계를 모델이 직접 학습할 수 있도록 하나의 변수로 제공합니다. 



In [ ]:
SERVICE_COLUMNS = [
    "Phone Service", "Multiple Lines", "Online Security", "Online Backup",
    "Device Protection Plan", "Premium Tech Support",
    "Streaming TV", "Streaming Movies", "Streaming Music",
]

# TODO: SERVICE_COLUMNS 중 "Yes"인 개수를 세어 df["TotalServices"]를 만드세요.
# 힌트: (df[SERVICE_COLUMNS] == "Yes").sum(axis=1)


# TODO: TotalServices별 이탈률을 total_services_churn 변수에 저장하세요.
# 힌트: df.groupby("TotalServices")["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(9, 5))
# TODO: total_services_churn을 이용해 TotalServices별 이탈률을 barplot으로 시각화하세요.

plt.title("TotalServices별 이탈률 (Churn Rate by Total Services)")
plt.xlabel("TotalServices (가입 서비스 개수)")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["TotalServices"]로 quick_evaluate를 호출해 성능을 확인하세요.


## Feature 3. StreamingCount / Feature 4. InternetCount

TotalServices는 모든 서비스를 하나로 합쳤지만, 서비스는 성격이 다른 두 그룹으로 나눌 수
있습니다.

- **엔터테인먼트 그룹**: Streaming TV, Streaming Movies, Streaming Music (최대 3개)
- **인터넷 보안/지원 그룹**: Online Security, Online Backup, Device Protection Plan,
  Premium Tech Support (최대 4개)

### 생성 이유
Finding 4에서 확인했듯 보안/지원 서비스는 이탈을 낮추는 효과가 뚜렷했습니다. 이 효과가
TotalServices에 섞여서 희석되지 않도록, 성격이 다른 두 그룹을 별도 변수로 분리합니다.


In [ ]:
STREAMING_COLUMNS = ["Streaming TV", "Streaming Movies", "Streaming Music"]
INTERNET_ADDON_COLUMNS = ["Online Security", "Online Backup", "Device Protection Plan", "Premium Tech Support"]

# TODO: STREAMING_COLUMNS 중 "Yes" 개수로 df["StreamingCount"]를,
# INTERNET_ADDON_COLUMNS 중 "Yes" 개수로 df["InternetCount"]를 만드세요.


print("StreamingCount별 이탈률:")
print(df.groupby("StreamingCount")["Churn"].agg(["mean", "count"]).round(4))

print("\nInternetCount별 이탈률:")
print(df.groupby("InternetCount")["Churn"].agg(["mean", "count"]).round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: StreamingCount별 이탈률을 streaming_churn에, InternetCount별 이탈률을 internet_churn에
# 저장하고 각각 axes[0], axes[1]에 barplot으로 그리세요.
# 힌트: streaming_churn = df.groupby("StreamingCount")["Churn"].mean()


axes[0].set_title("StreamingCount별 이탈률")
axes[0].set_xlabel("StreamingCount")
axes[0].set_ylabel("Churn Rate")

axes[1].set_title("InternetCount별 이탈률")
axes[1].set_xlabel("InternetCount")
axes[1].set_ylabel("Churn Rate")

plt.tight_layout()
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["TotalServices", "StreamingCount", "InternetCount"]로
# quick_evaluate를 호출해 성능을 확인하세요.


## Feature 5. AverageCharge (가입 기간 대비 평균 요금)

두 컬럼을 나누어 새로운 비율(Ratio) Feature를 만드는 방법과, 0으로 나누는 문제를
안전하게 처리하는 방법을 익힌다.

### 이론
`Total Charges`(총 요금)를 `Tenure in Months`(가입 기간)로 나누면 "평균적으로 매달
얼마를 냈는가"를 계산할 수 있습니다. 이는 프로모션 할인, 요금제 변경 이력 등이 반영된
"실질 평균 요금"이라는 점에서 `Monthly Charge`(현재 시점의 월 요금)와는 다른 정보를
담을 수 있습니다.

### 생성 이유
가입 기간 동안 요금이 바뀌었을 수 있는 고객을 구분해내기 위해 만듭니다.


In [ ]:
# TODO: Total Charges를 Tenure in Months로 나누어 df["AverageCharge"]를 만드세요.
# 0으로 나누는 문제를 방지하기 위해 Tenure in Months가 0인 경우를 NaN으로 바꾼 뒤 나누고,
# 그 결과를 다시 Monthly Charge 값으로 채우세요(fillna).
# 힌트: (df["Total Charges"] / df["Tenure in Months"].replace(0, np.nan)).fillna(df["Monthly Charge"])


print("AverageCharge와 Monthly Charge의 상관계수:", round(df["AverageCharge"].corr(df["Monthly Charge"]), 4))
df[["Monthly Charge", "AverageCharge"]].describe().round(2)


In [ ]:
plt.figure(figsize=(8, 5))
# TODO: Monthly Charge와 AverageCharge의 관계를 Churn Label로 색을 구분한 scatterplot으로 그리세요.
# 힌트: sns.scatterplot(x="Monthly Charge", y="AverageCharge", hue="Churn Label", data=df, alpha=0.4)

plt.title("Monthly Charge vs AverageCharge")
plt.xlabel("Monthly Charge (현재 월 요금)")
plt.ylabel("AverageCharge (실질 평균 요금)")
plt.show()


## Feature 6. HighChargeCustomer (고요금 고객 여부)

연속형 변수를 특정 기준(threshold)으로 이진화(Binarization)하는 방법과, 왜 반드시
"학습 데이터" 기준으로 기준값을 정해야 하는지 이해한다.

### 이론
가설 3(월 요금이 높을수록 이탈 확률이 높다)을 더 명확한 신호로 만들기 위해, 월 요금이
상위 25%(3사분위수)보다 높은 고객을 "고요금 고객"으로 표시합니다.

### 생성 이유
Monthly Charge를 연속값 그대로 두는 것보다, "비싸다/비싸지 않다"라는 이진 신호로 만들면
Logistic Regression 같은 모델이 임계값 효과를 더 쉽게 학습할 수 있습니다.


In [ ]:
# 탐색 단계에서는 전체 데이터의 3사분위수를 사용합니다.
# (Chapter 6 이후 공식 Pipeline에서는 Train 데이터 기준으로 다시 계산합니다)
# TODO: Monthly Charge의 상위 25%(3사분위수) 기준값을 high_charge_threshold에 저장하고,
# 이 값보다 큰 고객을 1, 아니면 0으로 하는 df["HighChargeCustomer"]를 만드세요.
# 힌트: df["Monthly Charge"].quantile(0.75)


# TODO: HighChargeCustomer별 이탈률을 high_charge_churn 변수에 저장하세요.
# 힌트: df.groupby("HighChargeCustomer")["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(6, 5))
# TODO: high_charge_churn을 이용해 HighChargeCustomer별 이탈률을 barplot으로 시각화하세요.

plt.title("HighChargeCustomer별 이탈률")
plt.xlabel("HighChargeCustomer (0=일반, 1=고요금)")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["HighChargeCustomer"]로 quick_evaluate를 호출해 성능을 확인하세요.


## Feature 7. ContractRisk (계약 + 가입 기간 위험 점수)

여러 위험 신호를 하나의 "점수(Score)"로 합치는 상호작용(Interaction) Feature를 설계할
수 있다.

### 이론
가설 7에서 예상했듯, Month-to-Month 계약과 신규 고객(6개월 이하)이라는 두 조건이
동시에 만족될 때 이탈률이 각각의 개별 효과를 합친 것보다 훨씬 크게 나타나는지
직접 계산해서 확인해봅니다. 두 조건을 개별 컬럼으로 각각 넣는 것보다, "위험 요인이
몇 개나 겹치는가"를 하나의 점수로 표현하면 모델이 상호작용을 더 쉽게 학습할 수 있습니다.

### 생성 이유
Month-to-Month 계약 여부(0/1)와 신규 고객 여부(0/1)를 더해서 0~2점 사이의 위험 점수를
만듭니다.


In [ ]:
# TODO: Contract가 "Month-to-Month"이면 1, 아니면 0인 df["IsMonthToMonth"]를 만드세요.
# TODO: Tenure in Months가 6 이하이면 1, 아니면 0인 df["IsNewCustomer"]를 만드세요.
# TODO: 두 값을 더해 df["ContractRisk"] (0~2점)를 만드세요.


# TODO: ContractRisk 점수별 이탈률을 contract_risk_churn 변수에 저장하세요.
# 힌트: df.groupby("ContractRisk")["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(7, 5))
# TODO: contract_risk_churn을 이용해 ContractRisk 점수별 이탈률을 barplot으로 시각화하세요.

plt.title("ContractRisk 점수별 이탈률")
plt.xlabel("ContractRisk (0=안정 ~ 2=고위험)")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["ContractRisk"]로 quick_evaluate를 호출해 성능을 확인하세요.


## Feature 8. SeniorLongTerm (고령 + 장기 고객 여부)

"주 효과(Main Effect)"와 "상호작용 효과(Interaction Effect)"가 다를 수 있다는 것을
직접 확인한다.

### 이론
Finding 7에서 고령 고객(Senior Citizen)의 이탈률이 41.68%로 매우 높다고 확인했습니다.
그런데 "고령이면서 동시에 가입 기간이 49개월 이상인 장기 고객"만 따로 떼어보면
어떨까요? 이 질문에 답하기 위한 Feature입니다.

### 생성 이유
"고령자는 무조건 위험하다"는 단순한 결론이 실제로도 맞는지, 가입 기간이라는 다른
변수와 결합했을 때도 유지되는지 검증하기 위해 만듭니다.


In [ ]:
# TODO: Tenure in Months가 49 이상이면 1인 df["IsLongTermCustomer"]를 만드세요.
# TODO: Senior Citizen이 "Yes"이면서 동시에 IsLongTermCustomer가 1인 경우 1인
# df["SeniorLongTerm"]을 만드세요 (AND 조건).


print("고령 고객 전체 이탈률:", round(df[df['Senior Citizen'] == 'Yes']['Churn'].mean(), 4))
print()
print("SeniorLongTerm 여부별 이탈률:")
print(df.groupby("SeniorLongTerm")["Churn"].agg(["mean", "count"]).round(4))


In [ ]:
# TODO: "고령 고객 전체"와 "고령+장기 고객(SeniorLongTerm=1)"의 이탈률을 비교하는
# DataFrame을 compare 변수로 만들고 barplot으로 시각화하세요.
# 힌트: compare = pd.DataFrame({"Group": [...], "ChurnRate": [...]})

plt.figure(figsize=(6, 5))

plt.title("고령 고객 전체 vs 고령+장기 고객 이탈률 비교")
plt.xlabel("")
plt.ylabel("Churn Rate")
plt.xticks(rotation=15)
plt.show()


## Feature 9. FamilyCustomer (가족 고객 여부)

OR 조건을 사용한 파생변수 생성 방법을 익힌다.

### 이론
가설 6에서 배우자(Married) 또는 부양가족(Dependents)이 있는 고객이 더 안정적이라고
확인했습니다. 두 조건 중 하나라도 해당되면 "가족 단위 고객"으로 볼 수 있습니다.

### 생성 이유
Married와 Dependents를 각각 보는 대신, "가족과 관련된 책임이 있는가"라는 하나의
질문으로 통합합니다.


In [ ]:
# TODO: Married가 "Yes"이거나 Dependents가 "Yes"이면 1인 df["FamilyCustomer"]를
# 만드세요 (OR 조건).
# 힌트: (df["Married"] == "Yes") | (df["Dependents"] == "Yes")


# TODO: FamilyCustomer별 이탈률을 family_churn 변수에 저장하세요.
# 힌트: df.groupby("FamilyCustomer")["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(6, 5))
# TODO: family_churn을 이용해 FamilyCustomer별 이탈률을 barplot으로 시각화하세요.

plt.title("FamilyCustomer별 이탈률")
plt.xlabel("FamilyCustomer (0=개인, 1=가족)")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["FamilyCustomer"]로 quick_evaluate를 호출해 성능을 확인하세요.


## Feature 10. CustomerValueTier (CLTV 기반 고객 가치 등급)

연속형 변수를 `qcut`(분위수 기준)으로 구간화하는 방법을 익힌다. `cut`(값 기준)과의
차이를 이해한다.

### 이론
`CLTV`(Customer Lifetime Value, 고객 생애 가치)는 이 고객이 장기적으로 회사에
가져다줄 것으로 예상되는 가치를 나타내는 지표입니다. 값의 범위가 넓고 분포가
고르지 않을 수 있으므로, 특정 값 기준(`cut`)이 아니라 **데이터 개수가 균등하게 나뉘는
분위수 기준(`qcut`)** 으로 3등급(Low, Medium, High)을 나눕니다.

### 생성 이유
가설 11(CLTV가 낮은 고객군의 이탈률이 더 높을 것이다)을 직접 검증하기 위해 만듭니다.


In [ ]:
# TODO: CLTV를 분위수 기준(qcut)으로 3등급(Low, Medium, High)으로 나누어
# df["CustomerValueTier"]를 만드세요.
# 힌트: pd.qcut(df["CLTV"], q=3, labels=["Low", "Medium", "High"])


# TODO: CustomerValueTier별 이탈률을 value_tier_churn 변수에 저장하세요 (observed=True 사용).
# 힌트: df.groupby("CustomerValueTier", observed=True)["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(6, 5))
# TODO: value_tier_churn을 이용해 CustomerValueTier별 이탈률을 barplot으로 시각화하세요.
# (order=["Low", "Medium", "High"])

plt.title("CustomerValueTier별 이탈률")
plt.xlabel("CustomerValueTier")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["CustomerValueTier"]로 quick_evaluate를 호출해 성능을 확인하세요.


## Feature 11. HighRiskCustomer (다중 위험 조건 결합 Feature)

지금까지 만든 여러 위험 신호를 하나의 "최종 고위험군" Feature로 결합하는 방법을
익힌다.

### 이론
지금까지 확인한 위험 신호를 정리하면 다음과 같습니다.

- Month-to-Month 계약 (IsMonthToMonth)
- 고요금 (HighChargeCustomer)
- 기술 지원 미가입 (Premium Tech Support = No)
- 신규 고객 (IsNewCustomer)

이 4가지 조건이 **모두** 겹치는 고객은 정말로 위험할까요? AND 조건으로 결합해서
직접 확인합니다.

### 생성 이유
개별 위험 신호는 각각 20~40% 수준의 이탈률을 보였지만, 여러 위험 신호가 동시에
나타나는 극단적인 고객군을 찾아내면 훨씬 더 정밀한 타겟팅이 가능합니다.


In [ ]:
# TODO: Premium Tech Support가 "No"이면 1인 df["NoTechSupport"]를 만드세요.

# TODO: IsMonthToMonth, HighChargeCustomer, NoTechSupport, IsNewCustomer가
# 모두 1인 경우에만 1인 df["HighRiskCustomer"]를 만드세요 (AND 조건 4개).


# TODO: HighRiskCustomer별 이탈률을 high_risk_churn 변수에 저장하세요.
# 힌트: df.groupby("HighRiskCustomer")["Churn"].agg(["mean", "count"]).round(4)


In [ ]:
plt.figure(figsize=(6, 5))
# TODO: high_risk_churn을 이용해 HighRiskCustomer별 이탈률을 barplot으로 시각화하세요.

plt.title("HighRiskCustomer별 이탈률")
plt.xlabel("HighRiskCustomer (0=일반, 1=4대 위험 조건 동시 충족)")
plt.ylabel("Churn Rate")
plt.show()


In [ ]:
# TODO: BASELINE_COLUMNS + ["HighRiskCustomer"]로 quick_evaluate를 호출해 성능을 확인하고,
# feature_experiment_df로 전체 실험 결과를 정리하세요.



# Chapter 5. Feature Selection (5단계 축소판)


- Feature Selection이 왜 Feature Engineering 다음에 필요한지 이해한다.
- 분산, 상관관계, 모델 기반 중요도라는 3가지 핵심 관점으로 Feature를 검증할 수 있다.

Chapter 4에서 원본 Feature에 11개의 파생변수를 추가하면서 전체 Feature 개수가 크게
늘어났습니다. Feature가 많다고 항상 좋은 것은 아닙니다.

- 정보량이 거의 없는 Feature(상수에 가까운 Feature)
- 서로 거의 같은 정보를 담은 중복 Feature
- 모델이 실제로 중요하게 사용하지 않는 Feature


```text
Step 1. Variance Threshold (분산)
    ↓
Step 2. Quasi Constant Feature (준상수)
    ↓
Step 3. Correlation Analysis (상관관계)
    ↓
Step 4. RandomForest Feature Importance (모델 기반 중요도)
    ↓
Step 5. Final Feature Selection Report (최종 종합)
```


In [ ]:
# Chapter 4에서 만든 모든 파생변수를 포함한 전체 Feature 목록을 정리합니다.
ENGINEERED_COLUMNS = [
    "TenureGroup", "TotalServices", "StreamingCount", "InternetCount",
    "AverageCharge", "HighChargeCustomer", "ContractRisk", "SeniorLongTerm",
    "FamilyCustomer", "CustomerValueTier", "HighRiskCustomer",
]

ALL_FEATURE_COLUMNS = BASELINE_COLUMNS + ENGINEERED_COLUMNS

print(f"Feature Selection 대상 전체 Feature 개수: {len(ALL_FEATURE_COLUMNS)}")

# TODO: ALL_FEATURE_COLUMNS를 One-Hot Encoding으로 숫자화한 X_fs를 만드세요.
# 힌트: pd.get_dummies(df[ALL_FEATURE_COLUMNS], drop_first=False)

y_fs = df["Churn"]

print("One-Hot Encoding 이후 Feature 개수:", X_fs.shape[1])


## Step 1. Variance Threshold

### 목적
분산(Variance)이 0인 Feature는 "모든 고객이 같은 값을 가진다"는 뜻입니다. 이런
Feature는 고객을 구분하는 데 아무 역할도 하지 못하므로 가장 먼저 걸러냅니다.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# TODO: VarianceThreshold(threshold=0.0)를 만들어 variance_selector에 저장하고 X_fs로 학습(fit)시키세요.
# 그 다음 분산이 0인 컬럼 목록을 zero_variance_columns에 저장하세요.
# 힌트: variance_selector.get_support()는 살아남은(분산이 0이 아닌) 컬럼이면 True를 반환합니다.
#       X_fs.columns[~variance_selector.get_support()]


print(f"분산이 0인 컬럼 개수: {len(zero_variance_columns)}")
print(zero_variance_columns)


## Step 2. Quasi Constant Feature (준상수 Feature)

### 목적
분산이 정확히 0은 아니지만, 특정 값이 거의 대부분(예: 99% 이상)을 차지하는
Feature를 찾습니다.

In [ ]:
def find_quasi_constant_columns(X, threshold):
    """가장 많이 등장하는 값의 비율이 (1 - threshold)보다 큰 컬럼을 찾는다."""
    quasi_constant_columns = []
    # TODO: X의 각 컬럼에 대해 최빈값 비율을 계산하고,
    # 비율이 (1 - threshold) 이상이면 quasi_constant_columns에 (컬럼명, 비율)을 추가하세요.
    # 힌트: X[col].value_counts(normalize=True).iloc[0]

    return quasi_constant_columns


for threshold in [0.01, 0.02, 0.05]:
    result = find_quasi_constant_columns(X_fs, threshold)
    print(f"threshold={threshold} -> 준상수 Feature {len(result)}개")
    for col, ratio in result:
        print(f"   {col} (최빈값 비율: {ratio})")


## Step 3. Correlation Analysis (상관관계 분석)

### 목적
서로 상관관계가 매우 높은 Feature 쌍을 찾아 중복 정보를 제거합니다.

In [ ]:
correlation_numeric_columns = numeric_columns + [
    "TotalServices", "StreamingCount", "InternetCount", "AverageCharge",
    "HighChargeCustomer", "ContractRisk", "SeniorLongTerm", "FamilyCustomer",
    "HighRiskCustomer",
]

# TODO: correlation_numeric_columns에 대한 상관관계 행렬(절댓값)을 계산해 fs_corr_matrix에
# 저장하고 heatmap으로 시각화하세요.
# 힌트: fs_corr_matrix = df[correlation_numeric_columns].corr().abs()

plt.figure(figsize=(12, 10))

plt.title("Feature Selection용 Correlation Heatmap (절댓값)")
plt.show()


In [ ]:
def find_highly_correlated_pairs(corr_matrix, threshold):
    pairs = []
    columns = corr_matrix.columns
    # TODO: 상관계수 행렬의 위쪽 삼각형(대각선 제외)을 순회하며
    # threshold보다 큰 상관계수를 가진 (컬럼A, 컬럼B, 상관계수) 쌍을 pairs에 추가하세요.

    return pairs


for threshold in [0.80, 0.85, 0.90]:
    pairs = find_highly_correlated_pairs(fs_corr_matrix, threshold)
    print(f"threshold={threshold} -> 상관계수가 threshold보다 높은 쌍 {len(pairs)}개")
    for a, b, v in pairs:
        print(f"   {a}  <->  {b}   (corr={v})")


## Step 4. RandomForest Feature Importance

### 목적
RandomForest는 학습 과정에서 각 Feature가 불순도(Impurity)를 얼마나 줄이는 데
기여했는지를 계산해서 `feature_importances_`로 제공합니다. 통계적 검정과 달리,
Feature들 사이의 비선형 상호작용까지 반영된 중요도입니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# TODO: X_fs, y_fs를 train_test_split으로 나누어 X_train_fs, X_test_fs, y_train_fs, y_test_fs를
# 만드세요 (test_size=0.2, random_state=RANDOM_STATE, stratify=y_fs).


# TODO: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)를 rf_for_selection에
# 만들어 X_train_fs, y_train_fs로 학습(fit)시키세요.


# TODO: rf_for_selection.feature_importances_로 rf_importance(내림차순 정렬된 Series,
# index=X_fs.columns)를 만드세요.


print("RandomForest Feature Importance Top 20:")
rf_importance.head(20)


In [ ]:
plt.figure(figsize=(9, 8))
# TODO: rf_importance 상위 20개를 가로 막대그래프(barh)로 시각화하세요.
# 힌트: rf_importance.head(20).sort_values().plot(kind="barh")

plt.title("RandomForest Feature Importance Top 20")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## Step 5. Final Feature Selection Report

### 목적
Step 1~4에서 확인한 결과를 종합하여 최종 Feature를 결정합니다.

### 방법
One-Hot Encoding으로 나뉜 컬럼들을 다시 **원본 컬럼 단위**로 묶어서 RandomForest
Importance를 합산합니다. (예: `Contract_Month-to-Month`, `Contract_One Year`,
`Contract_Two Year`의 importance를 모두 더해 `Contract`의 대표 importance로 사용)

In [ ]:
def original_column_of(encoded_name):
    """One-Hot Encoding된 컬럼 이름에서 원본 컬럼 이름을 찾는다."""
    for col in ALL_FEATURE_COLUMNS:
        if encoded_name == col or encoded_name.startswith(col + "_"):
            return col
    return encoded_name


# TODO: rf_importance를 DataFrame으로 변환해 importance_df를 만드세요.
# 컬럼명을 ["EncodedColumn", "Importance"]로 지정한 뒤,
# original_column_of()로 각 인코딩된 컬럼의 원본 컬럼명을 "OriginalColumn" 컬럼에 추가하세요.


# TODO: OriginalColumn 기준으로 groupby하여 Importance 합계를 구하고
# 내림차순 정렬한 뒤 original_level_importance에 저장하세요.


print(f"Feature Selection 대상 원본 컬럼 개수: {len(original_level_importance)}")
original_level_importance.head(30)


In [ ]:
plt.figure(figsize=(9, 10))
# TODO: original_level_importance 상위 24개를 가로 막대그래프(barh)로 시각화하세요.

plt.title("원본 컬럼 단위 RandomForest Importance 합계 (Top 24)")
plt.xlabel("Importance 합계")
plt.ylabel("Feature (원본 컬럼)")
plt.show()


In [ ]:
EXCLUDE_DUE_TO_REDUNDANCY = ["AverageCharge"]       # Step 3: Monthly Charge와 상관관계가 매우 높음
EXCLUDE_DUE_TO_BUSINESS_LOGIC = ["Population"]      # 위치 정보 뒷문 우려

TOP_N = 24

# TODO: original_level_importance 상위 TOP_N개 컬럼 중 EXCLUDE_DUE_TO_REDUNDANCY,
# EXCLUDE_DUE_TO_BUSINESS_LOGIC에 속하지 않는 컬럼만 남겨 FINAL_FEATURE_COLUMNS를 만드세요.


print(f"최종 선정된 Feature 개수: {len(FINAL_FEATURE_COLUMNS)}")
print(FINAL_FEATURE_COLUMNS)


# Chapter 6. Model Comparison


## 데이터 준비 (Train / Test Split)

모델을 학습하기 전에 데이터를 Train(학습용)과 Test(평가용)로 나눕니다. Test
데이터는 "아직 보지 못한 새로운 고객"을 흉내내는 역할을 하므로, 전처리 기준
(결측치 대체값, One-Hot Encoding 범주, 스케일링 기준)은 반드시 **Train 데이터에만**
`fit`해야 합니다. Test 데이터에 이 기준을 미리 반영하면(Data Leakage), 실제
서비스 환경보다 성능이 낙관적으로 보이는 착시가 생깁니다.

- `test_size=0.2`: 20%를 테스트용으로 남겨둡니다.
- `stratify=y`: Train/Test 모두 이탈 비율(73:27)을 동일하게 유지합니다.
- `random_state=42`: 실행할 때마다 같은 방식으로 나뉘도록 고정합니다.

In [ ]:
# Offer, Internet Type의 결측치는 2-4에서 확인했듯 "해당 사항 없음"을 의미하는
# 값입니다. 평균이나 최빈값이 아니라, 의미가 명확한 문자열로 채웁니다.
# TODO: df["Offer"]의 결측치를 "No Offer"로, df["Internet Type"]의 결측치를
# "No Internet"으로 채우세요 (fillna).


X = df[FINAL_FEATURE_COLUMNS].copy()
y = df["Churn"]


def is_categorical(series):
    return (
        pd.api.types.is_object_dtype(series)
        or pd.api.types.is_string_dtype(series)
        or isinstance(series.dtype, pd.CategoricalDtype)
    )


CATEGORICAL_FEATURES = [c for c in FINAL_FEATURE_COLUMNS if is_categorical(X[c])]
NUMERIC_FEATURES = [c for c in FINAL_FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES]

# TODO: X, y를 train_test_split으로 나누어 X_train, X_test, y_train, y_test를 만드세요.
# (test_size=0.2, random_state=RANDOM_STATE, stratify=y)


print("Train 크기:", X_train.shape, " Test 크기:", X_test.shape)
print("범주형 Feature :", CATEGORICAL_FEATURES)
print("숫자형 Feature :", NUMERIC_FEATURES)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# TODO: 숫자형 Feature 전처리 Pipeline(numeric_transformer)을 만드세요.
# - 결측치는 중앙값(median)으로 채우고(SimpleImputer)
# - StandardScaler로 스케일링합니다.


# TODO: 범주형 Feature 전처리 Pipeline(categorical_transformer)을 만드세요.
# - 결측치는 최빈값(most_frequent)으로 채우고(SimpleImputer)
# - OneHotEncoder(handle_unknown="ignore")로 인코딩합니다.


# TODO: ColumnTransformer로 두 Pipeline을 NUMERIC_FEATURES, CATEGORICAL_FEATURES에
# 각각 적용하는 preprocessor를 만드세요.


preprocessor


## 6-1. 3가지 모델 소개

| 모델 | 학습 방식 | 특징 |
|---|---|---|
| **Logistic Regression** | 입력 Feature의 선형 조합을 확률로 변환 | 가장 단순하고 해석하기 쉬운 Baseline. 스케일링에 민감함 |
| **Decision Tree** | 조건을 기준으로 데이터를 계속 분할 | 해석이 직관적이지만 단일 트리는 과적합되기 쉬움 |
| **Random Forest** | 여러 개의 Decision Tree를 앙상블(투표) | 과적합에 강하고, 비선형 관계와 상호작용을 잘 학습함 |


## 평가 지표 계산 함수 준비

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)


def evaluate_pipeline(pipeline, model_name, X_train, y_train, X_test, y_test):
    """Pipeline을 학습(fit)하고, Test 데이터로 예측한 뒤 주요 평가지표를 계산한다."""
    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)
    pred_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": model_name,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred), 4),
        "Recall": round(recall_score(y_test, pred), 4),
        "F1": round(f1_score(y_test, pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, pred_proba), 4),
    }

    print(f"=== {model_name} ===")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred))
    print("\nClassification Report:")
    print(classification_report(y_test, pred, target_names=["No Churn", "Churn"]))

    return metrics, pred, pred_proba, pipeline


model_comparison_results = []
fitted_pipelines = {}
model_predictions = {}

## 6-2. Logistic Regression

Logistic Regression은 각 Feature에 가중치(계수)를 곱해서 더한 값을 0~1 사이의
확률로 변환합니다. 계수를 통해 "어떤 Feature가 이탈 확률을 얼마나 높이는지/
낮추는지"를 직접 해석할 수 있다는 장점이 있습니다.

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: preprocessor와 LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)을
# 연결한 Pipeline을 logreg_pipeline으로 만드세요.
# 그 다음 evaluate_pipeline(logreg_pipeline, "Logistic Regression", X_train, y_train, X_test, y_test)를
# 호출해 (logreg_metrics, logreg_pred, logreg_proba, logreg_pipeline)으로 결과를 받으세요.


model_comparison_results.append(logreg_metrics)
fitted_pipelines["Logistic Regression"] = logreg_pipeline
model_predictions["Logistic Regression"] = (logreg_pred, logreg_proba)


## 6-3. Decision Tree

Decision Tree는 "Contract가 Month-to-Month인가?" 같은 질문을 반복하며 데이터를
분할합니다. 하나의 트리만 사용하면 Train 데이터에 과도하게 맞춰지는
과적합(Overfitting)이 발생하기 쉬워, `max_depth`로 깊이를 제한합니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# TODO: preprocessor와 DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)를
# 연결한 Pipeline을 tree_pipeline으로 만드세요.
# 그 다음 evaluate_pipeline(tree_pipeline, "Decision Tree", X_train, y_train, X_test, y_test)를
# 호출해 (tree_metrics, tree_pred, tree_proba, tree_pipeline)으로 결과를 받으세요.


model_comparison_results.append(tree_metrics)
fitted_pipelines["Decision Tree"] = tree_pipeline
model_predictions["Decision Tree"] = (tree_pred, tree_proba)


## 6-4. Random Forest

Random Forest는 수백 개의 Decision Tree를 각각 다른 데이터 샘플과 다른 Feature
조합으로 학습시킨 뒤, 다수결(투표)로 최종 예측을 결정합니다. 개별 트리의 과적합
문제가 평균화되어 완화됩니다. 현재 서비스(`train_model.py`)에서도 이미
RandomForestClassifier를 사용하고 있습니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: preprocessor와 RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)를
# 연결한 Pipeline을 rf_pipeline으로 만드세요.
# 그 다음 evaluate_pipeline(rf_pipeline, "Random Forest", X_train, y_train, X_test, y_test)를
# 호출해 (rf_metrics, rf_pred, rf_proba, rf_pipeline)으로 결과를 받으세요.


model_comparison_results.append(rf_metrics)
fitted_pipelines["Random Forest"] = rf_pipeline
model_predictions["Random Forest"] = (rf_pred, rf_proba)


## 6-5. 3개 모델 종합 비교

이제 Logistic Regression, Decision Tree, Random Forest 3개 모델의 성능을 표와
그래프로 자세히 비교합니다.

# Chapter 7. Final Model (최종 모델)

Chapter 6에서 Logistic Regression, Decision Tree, Random Forest 3개 모델을
실험했습니다.  이 프로젝트는 **Recall이 중요한 문제**이므로(Chapter 2-6), Recall을 최우선 기준으로
최종 모델을 선정합니다.

In [ ]:
# TODO: model_comparison_results를 DataFrame으로 변환해 model_comparison_df를 만드세요.


### 자세한 비교 해석

**1) Accuracy(정확도) 관점**
세 모델 모두 90% 중반대의 높은 Accuracy를 보이는 경향이 있습니다. 다만 앞서
확인했듯 이 데이터는 클래스 불균형(73:27)이 있으므로, Accuracy만으로 우열을
가리는 것은 위험합니다.

**2) Precision vs Recall 관점**
- Random Forest는 일반적으로 **Precision이 가장 높은** 경향을 보입니다. →
  "이탈할 것"이라고 예측했을 때 실제로 이탈하는 비율이 가장 높다는 뜻입니다.
- Logistic Regression과 Decision Tree는 상대적으로 **Recall**에서 강점을 보이는
  경우가 많습니다. → 실제 이탈 고객을 더 많이 찾아낸다는 뜻입니다.
- 이 프로젝트는 실제 이탈 고객을 놓치지 않는 것이 더 중요하므로(Chapter 2-6),
  Precision보다 **Recall과 F1을 우선** 살펴봐야 합니다.


In [ ]:
# TODO: model_comparison_df를 출력하고, Recall이 가장 높은 행을 best_recall_row로 찾으세요.
# 힌트: model_comparison_df["Recall"].idxmax()


In [ ]:
# TODO: best_recall_row에서 모델 이름을 FINAL_MODEL_NAME으로 추출하고,
# fitted_pipelines에서 해당 파이프라인을 final_pipeline으로 가져오세요.
# 힌트: best_recall_row["Model"], fitted_pipelines[FINAL_MODEL_NAME]

print(f"최종 선정 모델: {FINAL_MODEL_NAME}")


### 최종 모델 저장


In [ ]:
import joblib
import os

MODEL_SAVE_DIR = "../models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# 주의: 기존 서비스가 사용하는 customer_churn_model.pkl은 덮어쓰지 않고,
# 별도의 이름으로 저장합니다.
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, "customer_churn_simple_model.pkl")

# TODO: joblib.dump()로 final_pipeline을 MODEL_SAVE_PATH에 저장하세요.
# (try/except로 감싸서 오류가 나면 메시지를 출력하도록 작성해보세요)


# Chapter 8. Research Report (연구 보고서)

- 지금까지의 전체 분석 과정을 하나의 보고서로 요약할 수 있다.


In [ ]:
from IPython.display import Markdown, display

final_row = model_comparison_df[model_comparison_df["Model"] == FINAL_MODEL_NAME].iloc[0]

report_n_rows, report_n_cols = df.shape
report_churn_counts = df["Churn Label"].value_counts()
report_churn_ratio = df["Churn Label"].value_counts(normalize=True)
report_null_cols = df.isnull().sum()
report_null_cols = report_null_cols[report_null_cols > 0]

report = f"""
# Customer Churn AI - EDA / Feature Engineering / Model Comparison 연구 보고서

## 1. 프로젝트 개요

- 데이터셋: IBM Telco Customer Churn (`ai-server/data/ibm_telco.csv`, {report_n_rows}명, {report_n_cols}개 컬럼)
- Target: `Churn Label` (Yes=이탈 / No=잔류) -> 이진 분류(Binary Classification) 문제
- 이 보고서는 Feature Selection을 5단계로 진행하고, 모델 비교를 Logistic Regression / Decision Tree / Random Forest
  3개로 테스트하였다.

## 2. EDA 결과 요약

- 잔류(No) {report_churn_counts['No']}명({report_churn_ratio['No']*100:.2f}%), 이탈(Yes)
  {report_churn_counts['Yes']}명({report_churn_ratio['Yes']*100:.2f}%)로 클래스 불균형이 존재한다.
- 결측치가 있는 컬럼은 {len(report_null_cols)}개이다. ({', '.join(report_null_cols.index)})
  (TODO: 2-4에서 확인한 결측치의 의미를 적어보세요)
- (TODO: 2-11에서 정리한 Finding 중, 이탈률과 가장 뚜렷한 관계를 보인 변수를 적어보세요)

## 3. Hypothesis (가설)

Chapter 3에서 EDA 결과를 근거로 10개의 가설을 수립했다.
(TODO: 그중 가장 흥미로웠던 가설과 그 이유를 1~2문장으로 적어보세요)

## 4. Feature Engineering 결과

Chapter 4에서 11개의 파생변수(TenureGroup, TotalServices, StreamingCount,
InternetCount, AverageCharge, HighChargeCustomer, ContractRisk, SeniorLongTerm,
FamilyCustomer, CustomerValueTier, HighRiskCustomer)를 생성했다.
(TODO: 이 중 성능 향상에 가장 크게 기여한 Feature와 그 근거를 적어보세요)

## 5. Feature Selection 결과

Variance Threshold -> Quasi Constant -> Correlation Analysis -> RandomForest
Importance -> Final Report의 5단계를 적용한 결과, 원본 {len(BASELINE_COLUMNS)}개 +
파생 11개 = {len(ALL_FEATURE_COLUMNS)}개 Feature 중 최종 {len(FINAL_FEATURE_COLUMNS)}개를
선정했다.
(TODO: EXCLUDE_DUE_TO_REDUNDANCY, EXCLUDE_DUE_TO_BUSINESS_LOGIC으로 제외한 Feature와
그 이유를 적어보세요)

## 6. Model Comparison 결과

Logistic Regression, Decision Tree, Random Forest 3개 모델을 동일한 Train/Test
분할과 동일한 Pipeline으로 비교했다.
(TODO: 어떤 모델이 Recall 기준으로 가장 우수했는지, 그리고 그 이유로 추정되는 바를 적어보세요)

## 7. 최종 모델

- **최종 선정 모델**: {FINAL_MODEL_NAME}
- **선정 기준**: Recall 최우선
- **최종 성능**:
  - Accuracy: {final_row['Accuracy']}
  - Precision: {final_row['Precision']}
  - Recall: {final_row['Recall']}
  - F1: {final_row['F1']}
  - ROC-AUC: {final_row['ROC-AUC']}

## 8. FastAPI 적용 시 주의사항

1. 저장된 `customer_churn_simple_model.pkl`은 `FINAL_FEATURE_COLUMNS`
   ({len(FINAL_FEATURE_COLUMNS)}개)과 동일한 컬럼 구조의 입력을 받아야 한다.
2. `Offer`, `Internet Type`의 결측치를 각각 `"No Offer"`, `"No Internet"`으로
   채우는 로직을 예측 시점에도 동일하게 적용해야 한다.
3. `TenureGroup`, `ContractRisk`처럼 파생변수가 최종 목록에 포함되었다면, 예측
   요청이 들어올 때마다 동일한 공식으로 실시간 계산해야 한다.
4. 기존 서비스가 사용하는 `customer_churn_model.pkl`과 이번에 저장한
   `customer_churn_simple_model.pkl`은 입력 스키마가 다르므로, 실제 교체 적용 전
   `main.py`의 입력 스키마와 전처리 로직을 함께 수정해야 한다.

"""

display(Markdown(report))
